# Visualising regions

## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import cooler
import bioframe
import cooltools
from cooltools import expected_cis, expected_trans
from cooltools.lib import plotting
import cooler as clr
import h5py
import coolbox
from coolbox.api import *
from coolbox.utilities import GenomeRange
from pygenometracks import readBed, readGtf
from pygenometracks.tracks import BedTrack

#Things I added
import glob
from pathlib import Path

In [ ]:
plt.rcParams.update({"text.usetex": False})
plt.rc('pdf',fonttype = 42)

## Setup stuff for aesthetics

In [ ]:
# Define plotting aesthetics as global variables
HIGHLIGHT_PARAMS = {"color":"gray",
                   "alpha":0.05,
                   "border_line":False}

MICROC_PLOTTING_PARAMS = {"style":"matrix",
                          "transform":"log10",
                          "depth_ratio":"full",
                          "balance":True,
                          "cmap":"fall",
                          "max_value":-1.5}

SPACER_HEIGHT = 0.4
DEFAULT_COLOR_LIST = ["#FF0000", "#00FA00", "#0000FF", "#A0F000"]
PRO_SEQ_TRACK_HEIGHT = 1.2

ANNOTATIONS_PARAMS = [{"height":15},
                      {"file": "/mnt/md0/Miles/Analysis/mm39_NCBI_RefSeq_names_unique_nopreds.bed", # PATH TO BED FILE OF GENE ANNOTATIONS
                       "max_labels":1000,
                       "merge_transcripts":True, 
                       "gene_rows":30}]

def cat_prefix_to_list(prefix, file_list):
    """
    Helper function that concatenates a prefix to every element
    in a list. 
    """
    return [os.path.join(prefix, file) for file in file_list]

def make_region_plot(region, resolution, highlight_region_list, microc_file, microc_title,
                     chip_list_list, forward_proseq_list, reverse_proseq_list, condition_order, 
                     bigwig_bins=3600, highlight_buffer=1000, pro_bins=None):
    
    """
    Function that makes a plot of a region with Micro-C, ChIP-seq, and Pro-seq.
    
    region: (str) string that specifies region for plotting.
    resolution: (int) resolution of micro-c map used
    highlight_region_list: List of regions of format [chr, start, end] to highlight
    microc_file: (str) file path to micro-c data
    microc_title: (str) title for micro-c
    chip_list_list: List of lists containing paths to bigwigs
    forward_proseq_list: List of paths to forward strand pro-seq files
    reverse_proseq_list: List of paths to reverse strand pro-seq files
    condition_order: List indicating order of conditions to plot. 
    """
    
    heatmap = Cool(microc_file, resolution=resolution, **MICROC_PLOTTING_PARAMS) + Title(microc_title)
    frame = heatmap
    if pro_bins is None:
        pro_bins = bigwig_bins
    def iterate_over_datasets(frame):
        i = 0 
        for chip_list in chip_list_list:
            bigwig_list = make_bigwig_list(chip_list, region, condition_order, 
                                           DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=bigwig_bins)
            i += 1
            for bigwig in bigwig_list:
                frame += bigwig + Spacer(SPACER_HEIGHT)
        f_bigwig_list = make_bigwig_list(forward_proseq_list, region, condition_order, 
                                       DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=pro_bins,  
                                         track_height=PRO_SEQ_TRACK_HEIGHT)
        r_bigwig_list = make_bigwig_list(reverse_proseq_list, region, len(reverse_proseq_list)*[""], 
                               DEFAULT_COLOR_LIST[(i+1)%len(DEFAULT_COLOR_LIST)], bw_bins=pro_bins, 
                                         track_height=PRO_SEQ_TRACK_HEIGHT)
        r_bigwig_list = [r_bigwig + Inverted() for r_bigwig in r_bigwig_list]
        for f_bigwig, r_bigwig in zip(f_bigwig_list, r_bigwig_list):
                frame += f_bigwig + r_bigwig + Spacer(SPACER_HEIGHT)
        annotations = NewBed(*ANNOTATIONS_PARAMS)
        frame += annotations
        return frame
    
    # check if it needs a highlight
    if len(highlight_region_list) > 0:
        highlight_regions = []
        for highlight_region in highlight_region_list:
            highlight_region[1] -= highlight_buffer
            highlight_region[2] += highlight_buffer 
            highlight_regions.append(tuple(highlight_region))

        highlights = HighLights(highlight_regions, **HIGHLIGHT_PARAMS)
        with highlights:
            frame = iterate_over_datasets(frame)
    else:
        frame = iterate_over_datasets(frame)
    frame += XAxis()
    return frame

def auto_scale_bigwigs(bigwig_list, region, y_max=None, y_min=0):
    """
    Autoscales y axis of bigwigs in bigwig list based off values in `region`.
    """
    def get_max_y_value(bigwig_list, region):
        max_y_values = []
        for bigwig in bigwig_list:
            max_y_values.append(np.amax(bigwig.fetch_plot_data(GenomeRange(region))))

        y_max = round(max(max_y_values) * 1.05)
        return y_max
    
    if y_max is None:
        if type(region) == list:
            y_max_list = [get_max_y_value(bigwig_list, single_region) for single_region in region]
            y_max = max(y_max_list)
        else:
            y_max = get_max_y_value(bigwig_list, region)
                
    return [bigwig + MaxValue(y_max) + MinValue(y_min) for bigwig in bigwig_list]

def read_cooler(cooler_path, resolution=250):
    """
    Simple function wrapper to read cooler file at a particular resolution.
    """
    if clr.fileops.is_cooler(cooler_path):
        return clr.Cooler(cooler_path)
    elif clr.fileops.is_multires_file(cooler_path):
        return clr.Cooler(cooler_path + "::resolutions/" + str(int(resolution)))

# define new bed format
class NewBed(Track):
    """
    Custom pygenometracks class that allows for control over the aesthetics of gene annotations in
    region plots. 
    """
    def __init__(self, coolbox_prop_dict, pygenometracks_prop_dict):
        super().__init__(coolbox_prop_dict)  # init 
        self.pygenometracks_object = BedTrack(pygenometracks_prop_dict)

    def fetch_data(self, gr, **kwargs):
        pass

    def plot(self, ax, gr, **kwargs):
        self.pygenometracks_object.plot(ax, gr.chrom, gr.start, gr.end)
        ax.set_xlim([gr.start, gr.end])

In [ ]:
def make_bigwig_list(bigwigs, region, condition_order, color, bw_bins, track_height=1.6, autoscale=True, 
                     y_max=None):
    
    """
    Helper function for `make_region_plot` that converts file paths to bigwig plotting objects.
    """
    assert len(bigwigs) % len(condition_order) == 0
    bigwig_list = [BigWig(bigwig, number_of_bins=bw_bins) + Title(title) + Color(color) \
                            + TrackHeight(track_height) for bigwig, title in zip(bigwigs, condition_order)]
    if autoscale:
        bigwig_list = auto_scale_bigwigs(bigwig_list, region, y_max=y_max)
    return bigwig_list


### More stuff to change aesthetics

In [ ]:
class Spacer(Track):
    """
    The spacer track
    Does not have any real content, just used to split two tracks.

    Parameters
    ----------
    height : float, optional
        The height of Spacer track. (Default: Spacer.DEFAULT_HEIGHT)

    """

    DEFAULT_HEIGHT = 1

    def __init__(self, *args, **kwargs):
        properties_dict = {
            'height': Spacer.DEFAULT_HEIGHT,
        }
        if args:
            height = args[0]
            properties_dict['height'] = height
        properties_dict.update(kwargs)

        super().__init__(properties_dict)

    def fetch_data(self, gr: GenomeRange, **kwargs):
        pass

    def plot(self, ax, gr: GenomeRange, **kwargs):
        self.ax = ax
        ax.set_xlim(gr.start, gr.end)


class HLine(Track):
    """
    Horizontal line track.
    Used for add a horizontal line between two tracks.

    Parameters
    ----------
    line_width : float, optional
        (Default: HLine.DEFAULT_LINE_WIDTH)

    line_style : str, optional
        (Default: HLine.DEFAULT_LINE_STYLE)

    color : str, optional
        (Default: HLine.DEFAULT_COLOR)

    alpha : float, optional
        (Default: HLine.DEFAULT_ALPHA)

    height : float, optional
        The height of Spacer track. (Default: HLine.DEFAULT_HEIGHT)

    name : str, optional
        Track's name.
    """

    DEFAULT_PROPERTIES = {
        "line_width": 1.0,
        "line_style": '--',
        "height": 0.05,  # this just a empiric value,
        "color": '#000000',
        'alpha': 0.75
    }

    def __init__(self, **kwargs):
        properties = HLine.DEFAULT_PROPERTIES.copy()
        properties.update(kwargs)
        super().__init__(properties)

    def fetch_data(self, gr: GenomeRange, **kwargs):
        pass

    def plot(self, ax, gr: GenomeRange, **kwargs):
        self.ax = ax

        ax.set_xlim(gr.start, gr.end)
        ax.hlines(0, gr.start, gr.end,
                  linestyles=self.properties['line_style'],
                  linewidth=self.properties['line_width'],
                  colors=self.properties['color'],
                  alpha=self.properties['alpha'])


class XAxis(Track):
    """
    The x axis track.

    Parameters
    ----------
    height : float, optional
        Height of Spacer track. (Default: XAxis.DEFAULT_HEIGHT)

    fontsize : int, optional
        Font size of XAxis. (Default: XAxis.DEFAULT_FONTSIZE)

    where : {'top', 'bottom'}, optional
        The position of tick labels relative to the axis.
        (Default: 'bottom')

    name (str, optional):
        Track's name.
    """

    DEFAULT_PROPERTIES = {
        "where": "bottom",
        "height": 2,
        "fontsize": 15,
    }

    def __init__(self, **kwargs):
        properties = XAxis.DEFAULT_PROPERTIES.copy()
        properties.update(kwargs)
        super().__init__(properties)

    def fetch_data(self, gr: GenomeRange, **kwargs):
        pass

    def plot(self, ax, gr: GenomeRange, **kwargs):
        self.ax = ax

        ax.set_xlim(gr.start, gr.end)
        ticks = ax.get_xticks()

        if ticks[-1] - ticks[1] <= 1000:
            labels = ["{:.0f}".format((x))
                      for x in ticks]
            labels[-2] += " bp"
        elif ticks[-1] - ticks[1] <= 1e5:
            labels = ["{:,.0f}".format((x / 1e3))
                      for x in ticks]
            labels[-2] += " Kb"
        else:
            labels = ["{:,.1f} Mb".format((x / 1e6))
                      for x in ticks]
            #labels[-2] += " Mb"

        ax.axis["x"] = ax.new_floating_axis(0, 0.5)

        ax.axis["x"].axis.set_ticklabels(labels)
        ax.axis['x'].axis.set_tick_params(which='minor', bottom='on')

        ax.axis["x"].major_ticklabels.set(size=int(self.properties['fontsize']))

        if 'where' in self.properties and self.properties['where'] == 'top':
            ax.axis["x"].set_axis_direction("top")


class ChromName(Track):
    """
    Track for show chromosome name.

    Parameters
    ----------
    fontsize : float
        Font name to show.

    offset : float
        Offset ratio to the start position.
    """

    DEFAULT_PROPERTIES = {
        "fontsize": 50,
        "offset": 0.45,
    }

    def __init__(self, **kwargs):
        properties = ChromName.DEFAULT_PROPERTIES.copy()
        properties.update(kwargs)
        super().__init__(properties)

    def fetch_data(self, gr: GenomeRange, **kwargs):
        return gr.chrom  # return chromosome name

    def plot(self, ax, gr: GenomeRange, **kwargs):
        x = gr.start + self.properties['offset'] * (gr.end - gr.start)
        ax.text(x, 0, gr.chrom, fontsize=self.properties['fontsize'])
        ax.set_xlim(gr.start, gr.end)

### Redefine the Cool class to allow for plotting two contact matrices across the diagonal

In [ ]:
class Cool(HicMatBase):
    """
    Cool track from .cool, .mcool file.

    Parameters
    ----------
    file: str
        The file path of .cool, .mcool file

    balance: bool
        If use the balanced contact matrix.


    """

    DEFAULT_PROPERTIES = {
        'cmap': "JuiceBoxLike",
        'balance': True,
        "norm": "no",
        "transform": "log",
        "file_bottom": None
    }

    def __init__(self, file, **kwargs):
        properties = Cool.DEFAULT_PROPERTIES.copy()
        properties.update({
            'file': file,
            **kwargs
        })
        super().__init__(**properties)

    def fetch_data(self, gr: GenomeRange, gr2=None, **kwargs) -> np.ndarray:
        from coolbox.utilities.hic.wrap import CoolerWrap

        path = self.properties['file']
        binsize = kwargs.get('resolution', self.properties.get('resolution', 'auto'))
        wrap = CoolerWrap(path, balance=self.balance, binsize=binsize)
        arr = wrap.fetch(gr, gr2)

        self.fetched_binsize = wrap.fetched_binsize  # expose fetched binsize

        ## SUPPORT TWO COOLS ON TOP AND BOTTOM
        if self.properties['file_bottom'] is not None: # two matrices to be plotted in upper and lower columns
            print("HI")
            path2 = self.properties['file_bottom']
            wrap2 = CoolerWrap(path2, balance=self.balance, binsize=binsize)
            arr2 = wrap2.fetch(gr, gr2)

            combined_arr = np.zeros_like(arr2)

            # fill in the bottom half with arr2 and the top half with arr1
            upper_ind = np.triu_indices(np.shape(arr2)[0])
            lower_ind = np.tril_indices(np.shape(arr2)[0])

            combined_arr[lower_ind] = arr2[lower_ind]
            combined_arr[upper_ind] = arr[upper_ind]

            return self.fill_zero_nan(combined_arr)
        else:
            return self.fill_zero_nan(arr)

    def fetch_pixels(self, gr: GenomeRange, gr2=None, **kwargs):
        """
        Fetch the pixels table of upper triangle of the original contact matrix(not processed).

        Parameters
        ----------
        gr2 : GenomeRange, optional.

        balance : bool, optional
            balance matrix or not,
            default `self.is_balance`.

        resolution : {'auto', int}
            resolution of the data. for example 5000.
            'auto' for calculate resolution automatically.
            default 'auto'

        join : bool
            whether to expand the bin ID columns
            into (chrom, start, end).
            default True

        Returns
        -------
        pixels : pandas.core.frame.DataFrame
            Hi-C pixels table.
            The pixel table contains the non-zero upper triangle entries of the contact map.
        """
        from coolbox.utilities.hic.wrap import CoolerWrap

        path = self.properties['file']
        balance = kwargs.get('balance', self.is_balance)
        wrap = CoolerWrap(path, balance=balance, binsize=kwargs.get('resolution', 'auto'))

        return wrap.fetch_pixels(gr, gr2, join=kwargs.get('join', True))

    def infer_binsize(self, gr: GenomeRange, **kwargs) -> int:
        from coolbox.utilities.hic.wrap import CoolerWrap

        path = self.properties['file']
        wrap = CoolerWrap(path, balance=self.balance, binsize=kwargs.get('resolution', 'auto'))
        return wrap.infer_binsize(gr)

## File paths

### Merged replicates

In [ ]:
mcool_path = "/path/to/mcools/" # provide path to mcool files
mcool_files = glob.glob(f'{mcool_path}batch?/*.mcool', recursive=True)
#Also get the names:
mcool_names = [Path(mcool).stem.split('_merged')[0] for mcool in mcool_files]


### ChIP

In [ ]:
chip_path = "/path/to/chip/" # provide path to chip files 
chip_bigwigs = glob.glob(f'{chip_path}*.bw') # provide chip file names
#Also generate condition names for the bigwigs:
chip_bigwigs_names = [Path(bigwig).stem for bigwig in chip_bigwigs]


## Paper plots

### Published data comparison

In [ ]:
rhodes_mcool_path = "/path/to/mcools/from/Rhodes" # provide path to mcool files
rhodes_mcool_files = glob.glob(f'{rhodes_mcool_path}/*.mcool')
#Also get the names:
rhodes_mcool_names = [Path(mcool).stem.split('.')[0] for mcool in rhodes_mcool_files]
rhodes_mcool_names

## Regions

In [ ]:
bedcols = ['chrom', 'start', 'end']
cap_regs = pd.read_csv(r'/path/to/capture/regions.bed', 
                       sep='\t', names=bedcols, header=None)
cap_regs

#### mm10 version for Rhodes data

In [ ]:
bedcols = ['chrom', 'start', 'end', "gene"]
mm10_cap_regs = pd.read_csv(r'/path/to/capture/regions_mm10.bed', 
                       sep='\t', names=bedcols, header=None)
mm10_cap_regs

### Loci of interest

### Fig 1 example plots

In [ ]:
PRC12_mcool_files = [mcool for mcool in mcool_files if "dTAGRING1ABdTAGSUZ12" in mcool]
PRC12_mcool_names = [name for name in mcool_names if "dTAGRING1ABdTAGSUZ12" in name]

#Need to order these
PRC12_bigwigs_files = sorted([bw for bw in chip_bigwigs if "dTAGRING1ABdTAGSUZ12" in bw])
PRC12_bigwigs_names = sorted([bw for bw in chip_bigwigs_names if "dTAGRING1ABdTAGSUZ12" in bw])
#Note that sorted treats uppercase letters as before lowercase letters



condition_order = ['dTAGRING1ABdTAGSUZ12_UNT_RING1B_MERGED', 'dTAGRING1ABdTAGSUZ12_dTAG4h_RING1B_MERGED',
                   'dTAGRING1ABdTAGSUZ12_UNT_PCGF2_MERGED', 'dTAGRING1ABdTAGSUZ12_dTAG4h_PCGF2_MERGED',
                   'dTAGRING1ABdTAGSUZ12_UNT_PHC1_MERGED', 'dTAGRING1ABdTAGSUZ12_dTAG4h_PHC1_MERGED',
                   'dTAGRING1ABdTAGSUZ12_UNT_CBX2_MERGED', 'dTAGRING1ABdTAGSUZ12_dTAG4h_CBX2_MERGED',
                   'dTAGRING1ABdTAGSUZ12_UNT_CBX7_MERGED', 'dTAGRING1ABdTAGSUZ12_dTAG4h_CBX7_MERGED',
                   'dTAGRING1ABdTAGSUZ12_UNT_SUZ12_MERGED', 'dTAGRING1ABdTAGSUZ12_dTAG4h_SUZ12_MERGED']

PRC12_bigwigs_names

In [ ]:
RNA_path = "/path/to/ttseq/" # provide path to proseq files
RNA_forward_bigwigs = glob.glob(f'{RNA_path}*FORWARD*')
RNA_reverse_bigwigs = glob.glob(f'{RNA_path}*REVERSE*')
RNA_names = [Path(bigwig).stem.split('_FORWARD')[0] for bigwig in RNA_forward_bigwigs]

PRC12_RNA_names = sorted([bw for bw in RNA_names if "dRING1AB_dSUZ12" in bw])
PRC12_RNA_F = sorted([bw for bw in RNA_forward_bigwigs if "dRING1AB_dSUZ12" in bw])
PRC12_RNA_R = sorted([bw for bw in RNA_reverse_bigwigs if "dRING1AB_dSUZ12" in bw])
PRC12_RNA_names

#### Plot region 2 (Nkx2-2) and region 5 (Tbx5 - Lhx5) - but want a zoom for region 5

#### Region 2

In [ ]:
plot_dir = "/path/to/output/plot/"

region_num = 1
RCMC_res = 5000

In [ ]:
region = cap_regs.iloc[region_num]
region

In [ ]:
#RCMC
unt_files = [i for i in PRC12_mcool_files if "UNT" in i]
unt_names = [i for i in PRC12_mcool_names if "UNT" in i]
filenum = 0
mcool1 = unt_files[filenum]
name1 = unt_names[filenum]
if "UNT" in name1:
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

region = cap_regs.iloc[region_num]
region_str = f'{region[0]}:{region[1]}-{region[2]}' # provide region in format like chr15:99,921,434-100,682,822

#Reverse these
microc_file1 = mcool1
microc_file2 = mcool2
microc_title = f"{name1} vs {name2} region {region_num + 1} at {RCMC_res}"

heatmap = Cool(microc_file1, file_bottom=microc_file2, 
               resolution=RCMC_res, **MICROC_PLOTTING_PARAMS) + Title(microc_title)
frame = heatmap
frame += XAxis()

UNT_chip_bigwigs_names = [bigwigname for bigwigname in condition_order if "UNT" in bigwigname]
UNT_chip_list = [chip for chip in PRC12_bigwigs_files if "UNT" in chip]
UNT_chip_list_ordered = []
for chip_name in condition_order:
    UNT_chip_list_ordered += [chip for chip in UNT_chip_list if chip_name in chip]
    
bigwig_bins = 3600

i = 0
for UNT_chip, UNT_chip_bigwigs_name in zip(UNT_chip_list_ordered, UNT_chip_bigwigs_names):
    chip_list = [UNT_chip, UNT_chip.replace("UNT","dTAG4h")]
    name_list = [UNT_chip_bigwigs_name, UNT_chip_bigwigs_name.replace("UNT","dTAG4h")]
    bigwig_list = make_bigwig_list(chip_list, region_str, name_list, 
                               DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=bigwig_bins, autoscale=True)
    i += 1
    for bigwig in bigwig_list:
        frame += bigwig + Spacer(SPACER_HEIGHT)

pro_bins = 3600

forward_RNA_list = PRC12_RNA_F
reverse_RNA_list = PRC12_RNA_R

f_bigwig_list = make_bigwig_list(forward_RNA_list, region_str, PRC12_RNA_names, 
                               DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=pro_bins,  
                                 track_height=PRO_SEQ_TRACK_HEIGHT, autoscale=True)
r_bigwig_list = make_bigwig_list(reverse_RNA_list, region_str, PRC12_RNA_names, 
                       DEFAULT_COLOR_LIST[(i+1)%len(DEFAULT_COLOR_LIST)], bw_bins=pro_bins, 
                                 track_height=PRO_SEQ_TRACK_HEIGHT, autoscale=True)
r_bigwig_list = [r_bigwig + Inverted() for r_bigwig in r_bigwig_list]
for f_bigwig, r_bigwig in zip(f_bigwig_list, r_bigwig_list):
    f_max_y = f_bigwig.properties['max_value']
    r_max_y = r_bigwig.properties['max_value']
    #Apply a consistent scaling factor
    #Divide this factor by 5 to allow the Polycomb-repressed genes to be seen
    max_y = round(max(f_max_y, r_max_y) / 5)
    f_bigwig_maxnorm = f_bigwig + MaxValue(max_y)
    r_bigwig_maxnorm = r_bigwig + MaxValue(max_y)
    
    frame += f_bigwig + r_bigwig + Spacer(SPACER_HEIGHT)
        
annotations = NewBed(*ANNOTATIONS_PARAMS)
frame += annotations

fig = frame.plot(region_str)
fig.savefig(os.path.join(plot_dir, f"region{region_num + 1}_{name1} over {name2} at {RCMC_res} bp TEMP" + ".pdf"), format='pdf')
fig


#### Region 5

In [ ]:
region_num = 4
RCMC_res = 2000

In [ ]:
#Set up zoom for this region
region = cap_regs.iloc[region_num]
region_str = f'{region[0]}:119300000-120800000' # provide region in format like chr15:99,921,434-100,682,822
region_str


In [ ]:

#RCMC
unt_files = [i for i in PRC12_mcool_files if "UNT" in i]
unt_names = [i for i in PRC12_mcool_names if "UNT" in i]
filenum = 0
mcool1 = unt_files[filenum]
name1 = unt_names[filenum]
if "UNT" in name1:
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

# region = cap_regs.iloc[region_num]
# region_str = f'{region[0]}:{region[1]}-{region[2]}' # provide region in format like chr15:99,921,434-100,682,822

#Reverse these
microc_file1 = mcool1
microc_file2 = mcool2
microc_title = f"{name1} vs {name2} region {region_num + 1} at {RCMC_res}"

heatmap = Cool(microc_file1, file_bottom=microc_file2, 
               resolution=RCMC_res, **MICROC_PLOTTING_PARAMS) + Title(microc_title)
frame = heatmap
frame += XAxis()

UNT_chip_bigwigs_names = [bigwigname for bigwigname in condition_order if "UNT" in bigwigname]
UNT_chip_list = [chip for chip in PRC12_bigwigs_files if "UNT" in chip]
UNT_chip_list_ordered = []
for chip_name in condition_order:
    UNT_chip_list_ordered += [chip for chip in UNT_chip_list if chip_name in chip]
    
bigwig_bins = 3600

i = 0
for UNT_chip, UNT_chip_bigwigs_name in zip(UNT_chip_list_ordered, UNT_chip_bigwigs_names):
    chip_list = [UNT_chip, UNT_chip.replace("UNT","dTAG4h")]
    name_list = [UNT_chip_bigwigs_name, UNT_chip_bigwigs_name.replace("UNT","dTAG4h")]
    bigwig_list = make_bigwig_list(chip_list, region_str, name_list, 
                               DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=bigwig_bins, autoscale=True)
    i += 1
    for bigwig in bigwig_list:
        frame += bigwig + Spacer(SPACER_HEIGHT)

pro_bins = 3600

forward_RNA_list = PRC12_RNA_F
reverse_RNA_list = PRC12_RNA_R

f_bigwig_list = make_bigwig_list(forward_RNA_list, region_str, PRC12_RNA_names, 
                               DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=pro_bins,  
                                 track_height=PRO_SEQ_TRACK_HEIGHT, autoscale=True)
r_bigwig_list = make_bigwig_list(reverse_RNA_list, region_str, PRC12_RNA_names, 
                       DEFAULT_COLOR_LIST[(i+1)%len(DEFAULT_COLOR_LIST)], bw_bins=pro_bins, 
                                 track_height=PRO_SEQ_TRACK_HEIGHT, autoscale=True)
r_bigwig_list = [r_bigwig + Inverted() for r_bigwig in r_bigwig_list]
for f_bigwig, r_bigwig in zip(f_bigwig_list, r_bigwig_list):
    f_max_y = f_bigwig.properties['max_value']
    r_max_y = r_bigwig.properties['max_value']
    max_y = round(max(f_max_y, r_max_y) / 20)
    f_bigwig_maxnorm = f_bigwig + MaxValue(max_y)
    r_bigwig_maxnorm = r_bigwig + MaxValue(max_y)
    
    frame += f_bigwig + r_bigwig + Spacer(SPACER_HEIGHT)
        
annotations = NewBed(*ANNOTATIONS_PARAMS)
frame += annotations

fig = frame.plot(region_str)
fig.savefig(os.path.join(plot_dir, f"region{region_num + 1}_{name1} over {name2} at {RCMC_res} bp TEMP" + ".pdf"), format='pdf')
fig

#### Fig. 1 Plot zooms of specific interactions

Tbx5-Lhx5

In [ ]:
tbx5_loc = "chr5:119,900,000-120,000,000"
lhx5_loc = "chr5:120,521,000-120,621,000"


In [ ]:
ZOOM_PLOTTING_PARAMS = {"style":"matrix",
                          "transform":"log10",
                          "depth_ratio":"full",
                          "balance":True,
                          "cmap":"fall",
                          "max_value":-2}

In [ ]:
unt_files = [i for i in PRC12_mcool_files if "UNT" in i]
unt_names = [i for i in PRC12_mcool_names if "UNT" in i]
filenum = 0
mcool1 = unt_files[filenum]
name1 = unt_names[filenum]
if "UNT" in name1:
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

for mcool, name in zip([mcool1, mcool2], [name1, name2]):
    region_str1 = lhx5_loc
    region_str2 = tbx5_loc
    resolution = 2000


    frame1 = XAxis()
    frame2 = XAxis()
    sub_frames = {"left": frame2, "bottom": frame1}

    heatmap = Cool(mcool, resolution=resolution, min_value = -4.5, color_bar = False, **ZOOM_PLOTTING_PARAMS)
    # jv = JointView(heatmap, **sub_frames, space = 0, padding_left = 0)
    jv = JointView(heatmap)

    fig = jv.plot(region_str1, region_str2)
    
    fig.save(os.path.join(plot_dir, f"Tbx5Lhx5_zoom_" + name + "adj_contrast.svg"))


In [ ]:
unt_files = [i for i in PRC12_mcool_files if "UNT" in i]
unt_names = [i for i in PRC12_mcool_names if "UNT" in i]
filenum = 0
mcool1 = unt_files[filenum]
name1 = unt_names[filenum]
if "UNT" in name1:
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

for mcool, name in zip([mcool1, mcool2], [name1, name2]):
    region_str1 = lhx5_loc
    region_str2 = tbx5_loc
    resolution = 2000


    frame1 = XAxis()
    frame2 = XAxis()
    sub_frames = {"left": frame2, "bottom": frame1}

    heatmap = Cool(mcool, resolution=resolution, min_value = -4.5, color_bar = True, **ZOOM_PLOTTING_PARAMS)
    # jv = JointView(heatmap, **sub_frames, space = 0, padding_left = 0)
    jv = JointView(heatmap)

    fig = jv.plot(region_str1, region_str2)
    
    fig.save(os.path.join(plot_dir, f"Tbx5Lhx5_zoom_" + name + "adj_contrast_cbar.svg"))


Pax1-Foxa2

In [ ]:
pax1_loc = "chr2:147,157,000-147,257,000"
foxa2_loc = "chr2:147,832,000-147,942,000"

In [ ]:
unt_files = [i for i in PRC12_mcool_files if "UNT" in i]
unt_names = [i for i in PRC12_mcool_names if "UNT" in i]
filenum = 0
mcool1 = unt_files[filenum]
name1 = unt_names[filenum]
if "UNT" in name1:
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

for mcool, name in zip([mcool1, mcool2], [name1, name2]):
    region_str1 = foxa2_loc
    region_str2 = pax1_loc
    resolution = 2000


    frame1 = XAxis()
    frame2 = XAxis()
    sub_frames = {"left": frame2, "bottom": frame1}

    heatmap = Cool(mcool, resolution=resolution, min_value = -4.5, color_bar = False, **MICROC_PLOTTING_PARAMS)
    jv = JointView(heatmap)

    fig = jv.plot(region_str1, region_str2)
    
    fig.save(os.path.join(plot_dir, f"Pax1Foxa2_zoom_" + name + ".svg"))


Insm1-Foxa2

In [ ]:
insm1_loc = "chr2:146,009,000-146,119,000"
foxa2_loc = "chr2:147,832,000-147,942,000"

In [ ]:
ZOOM_PLOTTING_PARAMS = {"style":"matrix",
                          "transform":"log10",
                          "depth_ratio":"full",
                          "balance":True,
                          "cmap":"fall",
                          "max_value":-2.5}

In [ ]:
unt_files = [i for i in PRC12_mcool_files if "UNT" in i]
unt_names = [i for i in PRC12_mcool_names if "UNT" in i]

for mcool1, name1 in zip(unt_files, unt_names):
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

    for mcool, name in zip([mcool1, mcool2], [name1, name2]):
        region_str1 = foxa2_loc
        region_str2 = insm1_loc
        resolution = 2000


        frame1 = XAxis()
        frame2 = XAxis()
        sub_frames = {"left": frame2, "bottom": frame1}

        heatmap = Cool(mcool, resolution=resolution, min_value = -4.5, color_bar = False, **ZOOM_PLOTTING_PARAMS)
        jv = JointView(heatmap)

        fig = jv.plot(region_str1, region_str2)

        fig.save(os.path.join(plot_dir, f"Insm1Foxa2_zoom_" + name + ".svg"))


### Fig S2 Rhodes comparison

In [ ]:
filenum = 0

rhodes_res = 5000
RCMC_res = 2000
paldi_res = 2000

In [ ]:
PRC12_mcool_files = [mcool for mcool in mcool_files if "dTAGRING1ABdTAGSUZ12" in mcool]
PRC12_mcool_names = [name for name in mcool_names if "dTAGRING1ABdTAGSUZ12" in name]
PRC12_bigwigs_files = sorted([bw for bw in chip_bigwigs if "dTAGRING1ABdTAGSUZ12" in bw and "_RING1B_" in bw])
PRC12_bigwigs_names = sorted([bw for bw in chip_bigwigs_names if "dTAGRING1ABdTAGSUZ12" in bw and "_RING1B_" in bw])
#Note that sorted treats uppercase letters as before lowercase letters

PRC12_bigwigs_files

#### Also want ATAC for these comparisons



In [ ]:
ATAC_path = "/path/to/King/et/al/bigwigs/" # provide path to chip files 
ATAC_bigwigs = sorted(glob.glob(f'{ATAC_path}*.bw'), reverse = True) # provide chip file name
#Also generate condition names for the bigwigs:
ATAC_bigwigs_dKO = sorted([bw for bw in ATAC_bigwigs if "dKO" in bw])
ATAC_bigwigs_names = sorted([Path(bigwig).stem for bigwig in ATAC_bigwigs_dKO], reverse = True)

ATAC_bigwigs_names



#### Hoxd

In [ ]:
region_num = 0


In [ ]:
# Rhodes
unt_files = [i for i in rhodes_mcool_files if "Tir1" in i]
unt_names = [i for i in rhodes_mcool_names if "Tir1" in i]
mcool1 = unt_files[filenum]
name1 = unt_names[filenum]
if "Tir1" in name1:
    mcool2 = mcool1.replace("Tir1","Ring1B")
    name2 = name1.replace("Tir1","Ring1B")

region_mm10 = mm10_cap_regs.iloc[region_num]
region_str = f'{region_mm10[0]}:{region_mm10[1]}-{region_mm10[2]}' # provide region in format like chr15:99,921,434-100,682,822

#Reverse these
microc_title = f"{name1} vs {name2} region {region_num + 1} at {rhodes_res}"

heatmap = Cool(mcool1, file_bottom=mcool2, 
               resolution=resolution, **MICROC_PLOTTING_PARAMS) + Title(microc_title)
frame = heatmap
frame += XAxis()
fig = frame.plot(region_str)
fig.savefig(os.path.join(plot_dir, f"region{region_num + 1}_{name1} over {name2}" + ".pdf"), format='pdf')    


In [ ]:
#RCMC
unt_files = [i for i in PRC12_mcool_files if "UNT" in i]
unt_names = [i for i in PRC12_mcool_names if "UNT" in i]
filenum = 0
mcool1 = unt_files[filenum]
name1 = unt_names[filenum]
if "UNT" in name1:
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

region = cap_regs.iloc[region_num]
region_str = f'{region[0]}:{region[1]}-{region[2]}' # provide region in format like chr15:99,921,434-100,682,822

resolution = 5000
#Reverse these
microc_file1 = mcool1
microc_file2 = mcool2
microc_title = f"{name1} vs {name2} region {region_num + 1} at {RCMC_res}"

heatmap = Cool(microc_file1, file_bottom=microc_file2, 
               resolution=resolution, **MICROC_PLOTTING_PARAMS) + Title(microc_title)
frame = heatmap
frame += XAxis()

bigwig_bins = 3600

i = 0
bigwig_list = make_bigwig_list(PRC12_bigwigs_files, region_str, PRC12_bigwigs_names, 
                               DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=bigwig_bins, autoscale=True)
i += 1
for bigwig in bigwig_list:
    frame += bigwig + Spacer(SPACER_HEIGHT)
    
    
ATAC_list = make_bigwig_list(ATAC_bigwigs, region_str, ATAC_bigwigs_names, 
                               DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=bigwig_bins, autoscale=True)
i += 1
for bigwig in ATAC_list:
    frame += bigwig + Spacer(SPACER_HEIGHT)
    
annotations = NewBed(*ANNOTATIONS_PARAMS)
frame += annotations

fig = frame.plot(region_str)
fig.savefig(os.path.join(plot_dir, f"region{region_num + 1}_{name1} over {name2}" + "replot.pdf"), format='pdf')

#### Pax2

In [ ]:
region_num = 13

#Want to cut this region down a little, as the 5' end of it was poorly captured and looks like it maybe doesn't match mm10
region = cap_regs.iloc[region_num]
region["start"] = 43400000

#Get lifted over version of the new start
region_mm10 = mm10_cap_regs.iloc[region_num]
region_mm10["start"] = 43411561


In [ ]:
# Rhodes
unt_files = [i for i in rhodes_mcool_files if "Tir1" in i]
unt_names = [i for i in rhodes_mcool_names if "Tir1" in i]
mcool1 = unt_files[filenum]
name1 = unt_names[filenum]
if "Tir1" in name1:
    mcool2 = mcool1.replace("Tir1","Ring1B")
    name2 = name1.replace("Tir1","Ring1B")

region_mm10
region_str = f'{region_mm10[0]}:{region_mm10[1]}-{region_mm10[2]}' # provide region in format like chr15:99,921,434-100,682,822

#Reverse these
microc_title = f"{name1} vs {name2} region {region_num + 1} at {rhodes_res}"

heatmap = Cool(mcool1, file_bottom=mcool2, 
               resolution=resolution, **MICROC_PLOTTING_PARAMS) + Title(microc_title)
frame = heatmap
frame += XAxis()
fig = frame.plot(region_str)
fig.savefig(os.path.join(plot_dir, f"region{region_num + 1}_{name1} over {name2}" + ".pdf"), format='pdf')


In [ ]:
#RCMC
unt_files = [i for i in PRC12_mcool_files if "UNT" in i]
unt_names = [i for i in PRC12_mcool_names if "UNT" in i]
filenum = 0
mcool1 = unt_files[filenum]
name1 = unt_names[filenum]
if "UNT" in name1:
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

region_str = f'{region[0]}:{region[1]}-{region[2]}' # provide region in format like chr15:99,921,434-100,682,822

resolution = 5000
#Reverse these
microc_file1 = mcool1
microc_file2 = mcool2
microc_title = f"{name1} vs {name2} region {region_num + 1} at {RCMC_res}"

heatmap = Cool(microc_file1, file_bottom=microc_file2, 
               resolution=resolution, **MICROC_PLOTTING_PARAMS) + Title(microc_title)
frame = heatmap
frame += XAxis()

bigwig_bins = 3600

i = 0
bigwig_list = make_bigwig_list(PRC12_bigwigs_files, region_str, PRC12_bigwigs_names, 
                               DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=bigwig_bins, autoscale=True)
i += 1
for bigwig in bigwig_list:
    frame += bigwig + Spacer(SPACER_HEIGHT)
    
ATAC_list = make_bigwig_list(ATAC_bigwigs, region_str, ATAC_bigwigs_names, 
                               DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=bigwig_bins, autoscale=True)
i += 1
for bigwig in ATAC_list:
    frame += bigwig + Spacer(SPACER_HEIGHT)
annotations = NewBed(*ANNOTATIONS_PARAMS)
frame += annotations

fig = frame.plot(region_str)
fig.savefig(os.path.join(plot_dir, f"region{region_num + 1}_{name1} over {name2}" + "replot.pdf"), format='pdf')
fig

#### Also zoom-ins of these regions

In [ ]:
reg1_mm39 = "chr19:44500000-45300000"
reg2_mm39 = "chr2:74250000-74750000"
zoomregs = [reg1_mm39, reg2_mm39]

In [ ]:
#RCMC
unt_files = [i for i in PRC12_mcool_files if "UNT" in i]
unt_names = [i for i in PRC12_mcool_names if "UNT" in i]
filenum = 0
mcool1 = unt_files[filenum]
name1 = unt_names[filenum]
if "UNT" in name1:
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

regcount = 0
for region_str in zoomregs:
    resolution = 1000
    regcount += 1
    #Reverse these
    microc_file1 = mcool1
    microc_file2 = mcool2
    microc_title = f"{name1} vs {name2} region {region_num + 1} at {RCMC_res}"

    heatmap = Cool(microc_file1, file_bottom=microc_file2, 
                   resolution=resolution, **MICROC_PLOTTING_PARAMS) + Title(microc_title)
    frame = heatmap
    frame += XAxis()
    bigwig_bins = 3600

    i = 0
    bigwig_list = make_bigwig_list(PRC12_bigwigs_files, region_str, PRC12_bigwigs_names, 
                                   DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=bigwig_bins, autoscale=True)
    i += 1
    for bigwig in bigwig_list:
        frame += bigwig + Spacer(SPACER_HEIGHT)

    ATAC_list = make_bigwig_list(ATAC_bigwigs, region_str, ATAC_bigwigs_names, 
                                   DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=bigwig_bins, autoscale=True)
    i += 1
    for bigwig in ATAC_list:
        frame += bigwig + Spacer(SPACER_HEIGHT)
    annotations = NewBed(*ANNOTATIONS_PARAMS)
    frame += annotations

    fig = frame.plot(region_str)
    fig.savefig(os.path.join(plot_dir, f"region{regcount}_{name1} over {name2}" + "zoom.pdf"), format='pdf')
    fig

### Fig S3 Plots of other regions

In [ ]:
RCMC_res = 5000

In [ ]:
#### Only RING1B data from PRC1/2deg

In [ ]:
PRC12_mcool_files = [mcool for mcool in mcool_files if "dTAGRING1ABdTAGSUZ12" in mcool]
PRC12_mcool_names = [name for name in mcool_names if "dTAGRING1ABdTAGSUZ12" in name]
PRC12_bigwigs_files = sorted([bw for bw in chip_bigwigs if "dTAGRING1ABdTAGSUZ12" in bw and "_RING1B_" in bw])
PRC12_bigwigs_names = sorted([bw for bw in chip_bigwigs_names if "dTAGRING1ABdTAGSUZ12" in bw and "_RING1B_" in bw])
#Note that sorted treats uppercase letters as before lowercase letters

PRC12_bigwigs_files

In [ ]:
for regcount, region in cap_regs.iterrows():
    if regcount in [0, 1, 4, 13]:
        print("Region already plotted")
    elif regcount in [10, 11]:
        resolution = 2000
        print(f"Plotting region {region}")
        region_str = f'{region[0]}:{region[1]}-{region[2]}' # provide region in format like chr15:99,921,434-100,682,822

        #RCMC
        unt_files = [i for i in PRC12_mcool_files if "UNT" in i]
        unt_names = [i for i in PRC12_mcool_names if "UNT" in i]
        filenum = 0
        mcool1 = unt_files[filenum]
        name1 = unt_names[filenum]
        if "UNT" in name1:
            mcool2 = mcool1.replace("UNT","dTAG")
            name2 = name1.replace("UNT","dTAG")

        microc_file1 = mcool1
        microc_file2 = mcool2
        microc_title = f"{name1} vs {name2} region {region_num + 1} at {RCMC_res}"

        heatmap = Cool(microc_file1, file_bottom=microc_file2, 
                       resolution=resolution, **MICROC_PLOTTING_PARAMS) + Title(microc_title)
        frame = heatmap
        frame += XAxis()

        UNT_chip_bigwigs_names = [bigwigname for bigwigname in PRC12_bigwigs_names if "UNT" in bigwigname]
        UNT_chip_list = [chip for chip in PRC12_bigwigs_files if "UNT" in chip]

        bigwig_bins = 3600

        i = 0
        for UNT_chip, UNT_chip_bigwigs_name in zip(UNT_chip_list, UNT_chip_bigwigs_names):
            chip_list = [UNT_chip, UNT_chip.replace("UNT","dTAG4h")]
            name_list = [UNT_chip_bigwigs_name, UNT_chip_bigwigs_name.replace("UNT","dTAG4h")]
            bigwig_list = make_bigwig_list(chip_list, region_str, name_list, 
                                       DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=bigwig_bins, autoscale=True)
            i += 1
            for bigwig in bigwig_list:
                frame += bigwig + Spacer(SPACER_HEIGHT)

        annotations = NewBed(*ANNOTATIONS_PARAMS)
        frame += annotations

        fig = frame.plot(region_str)
        fig.savefig(os.path.join(plot_dir, f"region{regcount + 1}_{name1} over {name2} at {RCMC_res} bp" + ".pdf"), format='pdf')
        fig
    else:
        resolution = 5000
        print(f"Plotting region {region}")
        region_str = f'{region[0]}:{region[1]}-{region[2]}' # provide region in format like chr15:99,921,434-100,682,822

        #RCMC
        unt_files = [i for i in PRC12_mcool_files if "UNT" in i]
        unt_names = [i for i in PRC12_mcool_names if "UNT" in i]
        filenum = 0
        mcool1 = unt_files[filenum]
        name1 = unt_names[filenum]
        if "UNT" in name1:
            mcool2 = mcool1.replace("UNT","dTAG")
            name2 = name1.replace("UNT","dTAG")

        microc_file1 = mcool1
        microc_file2 = mcool2
        microc_title = f"{name1} vs {name2} region {region_num + 1} at {RCMC_res}"

        heatmap = Cool(microc_file1, file_bottom=microc_file2, 
                       resolution=RCMC_res, **MICROC_PLOTTING_PARAMS) + Title(microc_title)
        frame = heatmap
        frame += XAxis()

        UNT_chip_bigwigs_names = [bigwigname for bigwigname in PRC12_bigwigs_names if "UNT" in bigwigname]
        UNT_chip_list = [chip for chip in PRC12_bigwigs_files if "UNT" in chip]

        bigwig_bins = 3600

        i = 0
        for UNT_chip, UNT_chip_bigwigs_name in zip(UNT_chip_list, UNT_chip_bigwigs_names):
            chip_list = [UNT_chip, UNT_chip.replace("UNT","dTAG4h")]
            name_list = [UNT_chip_bigwigs_name, UNT_chip_bigwigs_name.replace("UNT","dTAG4h")]
            bigwig_list = make_bigwig_list(chip_list, region_str, name_list, 
                                       DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=bigwig_bins, autoscale=True)
            i += 1
            for bigwig in bigwig_list:
                frame += bigwig + Spacer(SPACER_HEIGHT)

        annotations = NewBed(*ANNOTATIONS_PARAMS)
        frame += annotations

        fig = frame.plot(region_str)
        fig.savefig(os.path.join(plot_dir, f"region{regcount + 1}_{name1} over {name2} at {RCMC_res} bp" + ".pdf"), format='pdf')
        fig



### Fig S4 example plots

In [ ]:
matches = ["dTAGSUZ12_"]

matched_mcool_files = sorted([mcool for mcool in mcool_files if any("/" + x in mcool for x in matches)])
matched_mcool_names = sorted([name for name in mcool_names if any(name.startswith(x) for x in matches)])

#For these ones, there isn't really any corresponding ChIP, so just plot the PRC1/2deg RING1B and SUZ12 tracks
chip_matches = ["RING1B_MERGED", "SUZ12_MERGED"]
matched_bigwigs_files = sorted([bw for bw in chip_bigwigs if "RING1AB" in bw and any(x in bw for x in chip_matches)])
matched_bigwigs_names = sorted([bw for bw in chip_bigwigs_names if "RING1AB" in bw and any(x in bw for x in chip_matches)])
#Note that sorted treats uppercase letters as before lowercase letters

matched_bigwigs_files

In [ ]:
#No TTseq for this

#### Plot region 2 (Nkx2-2) and region 5 (Tbx5 - Lhx5) - but want a zoom for region 5

#### Region 2

In [ ]:
region_num = 1
RCMC_res = 5000

In [ ]:
region = cap_regs.iloc[region_num]
region_str = f'{region[0]}:{region[1]}-{region[2]}' # provide region in format like chr15:99,921,434-100,682,822

#RCMC
for cell_line in matches:
    unt_files = [i for i in matched_mcool_files if "UNT" in i and cell_line in i]
    unt_names = [i for i in matched_mcool_names if "UNT" in i and cell_line in i]
    filenum = 0
    mcool1 = unt_files[filenum]
    name1 = unt_names[filenum]
    if "UNT" in name1:
        mcool2 = mcool1.replace("UNT","dTAG")
        name2 = name1.replace("UNT","dTAG")

    microc_file1 = mcool1
    microc_file2 = mcool2
    microc_title = f"{name1} vs {name2} region {region_num + 1} at {RCMC_res}"

    heatmap = Cool(microc_file1, file_bottom=microc_file2, 
                   resolution=RCMC_res, **MICROC_PLOTTING_PARAMS) + Title(microc_title)
    frame = heatmap
    frame += XAxis()

    UNT_chip_bigwigs_names = [bigwigname for bigwigname in matched_bigwigs_names if "UNT" in bigwigname]
    UNT_chip_list = [chip for chip in matched_bigwigs_files if "UNT" in chip]

    bigwig_bins = 3600
    #Only plot the WT
    i = 0
    for UNT_chip, UNT_chip_bigwigs_name in zip(UNT_chip_list, UNT_chip_bigwigs_names):
        bigwig_list = make_bigwig_list([UNT_chip], region_str, [UNT_chip_bigwigs_name], 
                                   DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=bigwig_bins, autoscale=True)
        i += 1
        for bigwig in bigwig_list:
            frame += bigwig + Spacer(SPACER_HEIGHT)

    annotations = NewBed(*ANNOTATIONS_PARAMS)
    frame += annotations

    fig = frame.plot(region_str)
    fig.savefig(os.path.join(plot_dir, f"region{region_num + 1}_{name1} over {name2} at {RCMC_res} bp" + ".pdf"), format='pdf')
    fig

### Fig. S4 Plot zooms of specific interactions

Pax1-Foxa2

In [ ]:
pax1_loc = "chr2:147,157,000-147,257,000"
foxa2_loc = "chr2:147,832,000-147,942,000"

In [ ]:

for cell_line in matches:
    unt_files = [i for i in matched_mcool_files if "UNT" in i and cell_line in i]
    unt_names = [i for i in matched_mcool_names if "UNT" in i and cell_line in i]
    #A for loop is not needed here, this could be written better...
    for mcool1, name1 in zip(unt_files, unt_names):
        mcool2 = mcool1.replace("UNT","dTAG")
        name2 = name1.replace("UNT","dTAG")
        print(name1 + " " + name2)
        print(mcool1 + " " + mcool2)

        for mcool, name in zip([mcool1, mcool2], [name1, name2]):
            region_str1 = foxa2_loc
            region_str2 = pax1_loc
            resolution = 2000


            frame1 = XAxis()
            frame2 = XAxis()
            sub_frames = {"left": frame2, "bottom": frame1}

            heatmap = Cool(mcool, resolution=resolution, min_value = -4.5, color_bar = False, **MICROC_PLOTTING_PARAMS)
            jv = JointView(heatmap)

            fig = jv.plot(region_str1, region_str2)

            fig.save(os.path.join(plot_dir, f"Pax1Foxa2_zoom_" + name + ".svg"))

Insm1-Foxa2

In [ ]:
insm1_loc = "chr2:146,009,000-146,119,000"
foxa2_loc = "chr2:147,832,000-147,942,000"

In [ ]:
ZOOM_PLOTTING_PARAMS = {"style":"matrix",
                          "transform":"log10",
                          "depth_ratio":"full",
                          "balance":True,
                          "cmap":"fall",
                          "max_value":-2.5}

In [ ]:
for cell_line in matches:
    unt_files = [i for i in matched_mcool_files if "UNT" in i and cell_line in i]
    unt_names = [i for i in matched_mcool_names if "UNT" in i and cell_line in i]
    #A for loop is not needed here, this could be written better...
    for mcool1, name1 in zip(unt_files, unt_names):
        mcool2 = mcool1.replace("UNT","dTAG")
        name2 = name1.replace("UNT","dTAG")
        print(name1 + " " + name2)
        print(mcool1 + " " + mcool2)
        for mcool1, name1 in zip(unt_files, unt_names):
            mcool2 = mcool1.replace("UNT","dTAG")
            name2 = name1.replace("UNT","dTAG")

            for mcool, name in zip([mcool1, mcool2], [name1, name2]):
                region_str1 = foxa2_loc
                region_str2 = insm1_loc
                resolution = 2000


                frame1 = XAxis()
                frame2 = XAxis()
                sub_frames = {"left": frame2, "bottom": frame1}

                heatmap = Cool(mcool, resolution=resolution, min_value = -4.5, color_bar = False, **ZOOM_PLOTTING_PARAMS)
                jv = JointView(heatmap)

                fig = jv.plot(region_str1, region_str2)

                fig.save(os.path.join(plot_dir, f"Insm1Foxa2_zoom_" + name + ".svg"))





### Fig 2 example plots

In [ ]:
matches = ["CBX2dTAG_", "dTAGPHC123_", "CBX7dTAG_"]

cPRC1_mcool_files = sorted([mcool for mcool in mcool_files if any(x in mcool for x in matches)])
cPRC1_mcool_names = sorted([name for name in mcool_names if any(x in name for x in matches)])

#Need to order these
cPRC1_bigwigs_files = sorted([bw for bw in chip_bigwigs if any(x in bw for x in matches)])
cPRC1_bigwigs_names = sorted([bw for bw in chip_bigwigs_names if any(x in bw for x in matches)])
#Note that sorted treats uppercase letters as before lowercase letters

cPRC1_mcool_files


In [ ]:
cPRC1_bigwigs_names = sorted([bw for bw in chip_bigwigs_names if "CBX2dTAG_" in bw and "UNT" in bw])
cPRC1_bigwigs_names


In [ ]:
RNA_forward_bigwigs = glob.glob(f'{RNA_path}*FORWARD*')
RNA_reverse_bigwigs = glob.glob(f'{RNA_path}*REVERSE*')
RNA_names = [Path(bigwig).stem.split('_FORWARD')[0] for bigwig in RNA_forward_bigwigs]

cPRC1_RNA_names = sorted([bw for bw in RNA_names if any(x in bw for x in matches)])
cPRC1_RNA_F = sorted([bw for bw in RNA_forward_bigwigs if any(x in bw for x in matches)])
cPRC1_RNA_R = sorted([bw for bw in RNA_reverse_bigwigs if any(x in bw for x in matches)])
cPRC1_RNA_names

#### Plot region 2 (Nkx2-2)

#### Region 2

In [ ]:
region_num = 1
RCMC_res = 5000

In [ ]:
region = cap_regs.iloc[region_num]
region_str = f'{region[0]}:{region[1]}-{region[2]}' # provide region in format like chr15:99,921,434-100,682,822

#RCMC
for cell_line in matches:
    unt_files = [i for i in cPRC1_mcool_files if "UNT" in i and cell_line in i]
    unt_names = [i for i in cPRC1_mcool_names if "UNT" in i and cell_line in i]
    filenum = 0
    mcool1 = unt_files[filenum]
    name1 = unt_names[filenum]
    if "UNT" in name1:
        mcool2 = mcool1.replace("UNT","dTAG")
        name2 = name1.replace("UNT","dTAG")

    microc_file1 = mcool1
    microc_file2 = mcool2
    microc_title = f"{name1} vs {name2} region {region_num + 1} at {RCMC_res}"

    heatmap = Cool(microc_file1, file_bottom=microc_file2, 
                   resolution=RCMC_res, **MICROC_PLOTTING_PARAMS) + Title(microc_title)
    frame = heatmap
    frame += XAxis()

    UNT_chip_bigwigs_names = [bigwigname for bigwigname in chip_bigwigs_names if "UNT" in bigwigname and cell_line in bigwigname]
    UNT_chip_list = [chip for chip in chip_bigwigs if "UNT" in chip and cell_line in chip]

    bigwig_bins = 3600

    i = 0
    for UNT_chip, UNT_chip_bigwigs_name in zip(UNT_chip_list, UNT_chip_bigwigs_names):
        chip_list = [UNT_chip, UNT_chip.replace("UNT","dTAG4h")]
        name_list = [UNT_chip_bigwigs_name, UNT_chip_bigwigs_name.replace("UNT","dTAG4h")]
        bigwig_list = make_bigwig_list(chip_list, region_str, name_list, 
                                   DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=bigwig_bins, autoscale=True)
        i += 1
        for bigwig in bigwig_list:
            frame += bigwig + Spacer(SPACER_HEIGHT)

    pro_bins = 3600

    forward_RNA_list = sorted([bw for bw in RNA_forward_bigwigs if cell_line in bw])
    reverse_RNA_list = sorted([bw for bw in RNA_reverse_bigwigs if cell_line in bw])
    RNA_name_list = sorted([bw for bw in RNA_names if cell_line in bw])

    f_bigwig_list = make_bigwig_list(forward_RNA_list, region_str, RNA_name_list, 
                                   DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=pro_bins,  
                                     track_height=PRO_SEQ_TRACK_HEIGHT, autoscale=True)
    r_bigwig_list = make_bigwig_list(reverse_RNA_list, region_str, RNA_name_list, 
                           DEFAULT_COLOR_LIST[(i+1)%len(DEFAULT_COLOR_LIST)], bw_bins=pro_bins, 
                                     track_height=PRO_SEQ_TRACK_HEIGHT, autoscale=True)
    r_bigwig_list = [r_bigwig + Inverted() for r_bigwig in r_bigwig_list]
    for f_bigwig, r_bigwig in zip(f_bigwig_list, r_bigwig_list):
        f_max_y = f_bigwig.properties['max_value']
        r_max_y = r_bigwig.properties['max_value']
        #Apply a consistent scaling factor
        #Divide this factor by 5 to allow the Polycomb-repressed genes to be seen
        max_y = round(max(f_max_y, r_max_y) / 5)
        f_bigwig_maxnorm = f_bigwig + MaxValue(max_y)
        r_bigwig_maxnorm = r_bigwig + MaxValue(max_y)

        frame += f_bigwig + r_bigwig + Spacer(SPACER_HEIGHT)

    annotations = NewBed(*ANNOTATIONS_PARAMS)
    frame += annotations

    fig = frame.plot(region_str)
    fig.savefig(os.path.join(plot_dir, f"region{region_num + 1}_{name1} over {name2} at {RCMC_res} bp TEMP" + ".pdf"), format='pdf')
    fig



#### Fig. 2 Plot zooms of specific interactions

Tbx5-Lhx5

In [ ]:
tbx5_loc = "chr5:119,900,000-120,000,000"
lhx5_loc = "chr5:120,521,000-120,621,000"


In [ ]:
ZOOM_PLOTTING_PARAMS = {"style":"matrix",
                          "transform":"log10",
                          "depth_ratio":"full",
                          "balance":True,
                          "cmap":"fall",
                          "max_value":-2}

In [ ]:
unt_files = [i for i in cPRC1_mcool_files if "UNT" in i]
unt_names = [i for i in cPRC1_mcool_names if "UNT" in i]
filenum = 0

for mcool1, name1 in zip(unt_files, unt_names):
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

    for mcool, name in zip([mcool1, mcool2], [name1, name2]):
        region_str1 = lhx5_loc
        region_str2 = tbx5_loc
        resolution = 2000


        frame1 = XAxis()
        frame2 = XAxis()
        sub_frames = {"left": frame2, "bottom": frame1}

        heatmap = Cool(mcool, resolution=resolution, min_value = -4.5, color_bar = False, **ZOOM_PLOTTING_PARAMS)
        jv = JointView(heatmap)

        fig = jv.plot(region_str1, region_str2)

        fig.save(os.path.join(plot_dir, f"Tbx5Lhx5_zoom_" + name + "adj_contrast.svg"))


Pax1-Foxa2

In [ ]:
pax1_loc = "chr2:147,157,000-147,257,000"
foxa2_loc = "chr2:147,832,000-147,942,000"

In [ ]:
unt_files = [i for i in cPRC1_mcool_files if "UNT" in i]
unt_names = [i for i in cPRC1_mcool_names if "UNT" in i]

for mcool1, name1 in zip(unt_files, unt_names):
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

    for mcool, name in zip([mcool1, mcool2], [name1, name2]):
        region_str1 = foxa2_loc
        region_str2 = pax1_loc
        resolution = 2000


        frame1 = XAxis()
        frame2 = XAxis()
        sub_frames = {"left": frame2, "bottom": frame1}

        heatmap = Cool(mcool, resolution=resolution, min_value = -4.5, color_bar = False, **MICROC_PLOTTING_PARAMS)
        jv = JointView(heatmap)

        fig = jv.plot(region_str1, region_str2)

        fig.save(os.path.join(plot_dir, f"Pax1Foxa2_zoom_" + name + ".svg"))





Insm1-Foxa2

In [ ]:
insm1_loc = "chr2:146,009,000-146,119,000"
foxa2_loc = "chr2:147,832,000-147,942,000"

In [ ]:
ZOOM_PLOTTING_PARAMS = {"style":"matrix",
                          "transform":"log10",
                          "depth_ratio":"full",
                          "balance":True,
                          "cmap":"fall",
                          "max_value":-2.5}

In [ ]:
unt_files = [i for i in cPRC1_mcool_files if "UNT" in i]
unt_names = [i for i in cPRC1_mcool_names if "UNT" in i]

for mcool1, name1 in zip(unt_files, unt_names):
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

    for mcool, name in zip([mcool1, mcool2], [name1, name2]):
        region_str1 = foxa2_loc
        region_str2 = insm1_loc
        resolution = 2000


        frame1 = XAxis()
        frame2 = XAxis()
        sub_frames = {"left": frame2, "bottom": frame1}

        heatmap = Cool(mcool, resolution=resolution, min_value = -4.5, color_bar = True, **ZOOM_PLOTTING_PARAMS)
        jv = JointView(heatmap)

        fig = jv.plot(region_str1, region_str2)

        fig.save(os.path.join(plot_dir, f"Insm1Foxa2_zoom_" + name + "cbar.svg"))


### Fig 4 example plots

In [ ]:
matches = ["PCGF24dTAG_"]

cPRC1_mcool_files = sorted([mcool for mcool in mcool_files if any("/" + x in mcool for x in matches)])
cPRC1_mcool_names = sorted([name for name in mcool_names if name.startswith(tuple(matches))])

#Need to order these
cPRC1_bigwigs_files = sorted([bw for bw in chip_bigwigs if any(x in bw for x in matches)])
cPRC1_bigwigs_names = sorted([bw for bw in chip_bigwigs_names if any(x in bw for x in matches)])
#Note that sorted treats uppercase letters as before lowercase letters

cPRC1_mcool_files

In [ ]:
RNA_forward_bigwigs = glob.glob(f'{RNA_path}*FORWARD*')
RNA_reverse_bigwigs = glob.glob(f'{RNA_path}*REVERSE*')
RNA_names = [Path(bigwig).stem.split('_FORWARD')[0] for bigwig in RNA_forward_bigwigs]

cPRC1_RNA_names = sorted([bw for bw in RNA_names if bw.startswith(tuple(matches))])
cPRC1_RNA_F = sorted([bw for bw in RNA_forward_bigwigs if any("/" + x in bw for x in matches)])
cPRC1_RNA_R = sorted([bw for bw in RNA_reverse_bigwigs if any("/" + x in bw for x in matches)])
cPRC1_RNA_F

#### Plot region 2 (Nkx2-2)

#### Region 2

In [ ]:
region_num = 1
RCMC_res = 5000

In [ ]:
region = cap_regs.iloc[region_num]
region_str = f'{region[0]}:{region[1]}-{region[2]}' # provide region in format like chr15:99,921,434-100,682,822

#RCMC
for cell_line in matches:
    unt_files = [i for i in cPRC1_mcool_files if "UNT" in i and cell_line in i]
    unt_names = [i for i in cPRC1_mcool_names if "UNT" in i and cell_line in i]
    filenum = 0
    mcool1 = unt_files[filenum]
    name1 = unt_names[filenum]
    if "UNT" in name1:
        mcool2 = mcool1.replace("UNT","dTAG")
        name2 = name1.replace("UNT","dTAG")

    microc_file1 = mcool1
    microc_file2 = mcool2
    microc_title = f"{name1} vs {name2} region {region_num + 1} at {RCMC_res}"

    heatmap = Cool(microc_file1, file_bottom=microc_file2, 
                   resolution=RCMC_res, **MICROC_PLOTTING_PARAMS) + Title(microc_title)
    frame = heatmap
    frame += XAxis()

    UNT_chip_bigwigs_names = [bigwigname for bigwigname in chip_bigwigs_names if "UNT" in bigwigname and cell_line in bigwigname]
    UNT_chip_list = [chip for chip in chip_bigwigs if "UNT" in chip and cell_line in chip]

    bigwig_bins = 3600

    i = 0
    for UNT_chip, UNT_chip_bigwigs_name in zip(UNT_chip_list, UNT_chip_bigwigs_names):
        chip_list = [UNT_chip, UNT_chip.replace("UNT","dTAG4h")]
        name_list = [UNT_chip_bigwigs_name, UNT_chip_bigwigs_name.replace("UNT","dTAG4h")]
        bigwig_list = make_bigwig_list(chip_list, region_str, name_list, 
                                   DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=bigwig_bins, autoscale=True)
        i += 1
        for bigwig in bigwig_list:
            frame += bigwig + Spacer(SPACER_HEIGHT)

    pro_bins = 3600

    forward_RNA_list = sorted([bw for bw in RNA_forward_bigwigs if "/" + cell_line in bw])
    print(forward_RNA_list)
    reverse_RNA_list = sorted([bw for bw in RNA_reverse_bigwigs if "/" + cell_line in bw])
    print(reverse_RNA_list)
    RNA_name_list = sorted([bw for bw in RNA_names if bw.startswith(cell_line)])
    print(RNA_name_list)

    f_bigwig_list = make_bigwig_list(forward_RNA_list, region_str, RNA_name_list, 
                                   DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=pro_bins,  
                                     track_height=PRO_SEQ_TRACK_HEIGHT, autoscale=True)
    r_bigwig_list = make_bigwig_list(reverse_RNA_list, region_str, RNA_name_list, 
                           DEFAULT_COLOR_LIST[(i+1)%len(DEFAULT_COLOR_LIST)], bw_bins=pro_bins, 
                                     track_height=PRO_SEQ_TRACK_HEIGHT, autoscale=True)
    r_bigwig_list = [r_bigwig + Inverted() for r_bigwig in r_bigwig_list]
    for f_bigwig, r_bigwig in zip(f_bigwig_list, r_bigwig_list):
        f_max_y = f_bigwig.properties['max_value']
        r_max_y = r_bigwig.properties['max_value']
        #Apply a consistent scaling factor
        #Divide this factor by 5 to allow the Polycomb-repressed genes to be seen
        max_y = round(max(f_max_y, r_max_y) / 5)
        f_bigwig_maxnorm = f_bigwig + MaxValue(max_y)
        r_bigwig_maxnorm = r_bigwig + MaxValue(max_y)

        frame += f_bigwig + r_bigwig + Spacer(SPACER_HEIGHT)

    annotations = NewBed(*ANNOTATIONS_PARAMS)
    frame += annotations

    fig = frame.plot(region_str)
    fig.savefig(os.path.join(plot_dir, f"region{region_num + 1}_{name1} over {name2} at {RCMC_res} bp TEMP" + ".pdf"), format='pdf')
    fig



#### Fig. 4 Plot zooms of specific interactions

Tbx5-Lhx5

In [ ]:
tbx5_loc = "chr5:119,900,000-120,000,000"
lhx5_loc = "chr5:120,521,000-120,621,000"

In [ ]:
ZOOM_PLOTTING_PARAMS = {"style":"matrix",
                          "transform":"log10",
                          "depth_ratio":"full",
                          "balance":True,
                          "cmap":"fall",
                          "max_value":-2}

In [ ]:
unt_files = [i for i in cPRC1_mcool_files if "UNT" in i]
unt_names = [i for i in cPRC1_mcool_names if "UNT" in i]
filenum = 0

for mcool1, name1 in zip(unt_files, unt_names):
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

    for mcool, name in zip([mcool1, mcool2], [name1, name2]):
        region_str1 = lhx5_loc
        region_str2 = tbx5_loc
        resolution = 2000


        frame1 = XAxis()
        frame2 = XAxis()
        sub_frames = {"left": frame2, "bottom": frame1}

        heatmap = Cool(mcool, resolution=resolution, min_value = -4.5, color_bar = False, **ZOOM_PLOTTING_PARAMS)
        jv = JointView(heatmap)

        fig = jv.plot(region_str1, region_str2)

        fig.save(os.path.join(plot_dir, f"Tbx5Lhx5_zoom_" + name + "adj_contrast.svg"))


Pax1-Foxa2

In [ ]:
pax1_loc = "chr2:147,162,000-147,262,000"
foxa2_loc = "chr2:147,832,000-147,942,000"

In [ ]:
unt_files = [i for i in cPRC1_mcool_files if "UNT" in i]
unt_names = [i for i in cPRC1_mcool_names if "UNT" in i]

for mcool1, name1 in zip(unt_files, unt_names):
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

    for mcool, name in zip([mcool1, mcool2], [name1, name2]):
        region_str1 = foxa2_loc
        region_str2 = pax1_loc
        resolution = 2000


        frame1 = XAxis()
        frame2 = XAxis()
        sub_frames = {"left": frame2, "bottom": frame1}

        heatmap = Cool(mcool, resolution=resolution, min_value = -4.5, color_bar = False, **MICROC_PLOTTING_PARAMS)
        jv = JointView(heatmap)

        fig = jv.plot(region_str1, region_str2)

        fig.save(os.path.join(plot_dir, f"Pax1Foxa2_zoom_" + name + ".svg"))


Insm1-Foxa2

In [ ]:
insm1_loc = "chr2:146,009,000-146,119,000"
foxa2_loc = "chr2:147,832,000-147,942,000"

In [ ]:
ZOOM_PLOTTING_PARAMS = {"style":"matrix",
                          "transform":"log10",
                          "depth_ratio":"full",
                          "balance":True,
                          "cmap":"fall",
                          "max_value":-2.5}

In [ ]:
unt_files = [i for i in cPRC1_mcool_files if "UNT" in i]
unt_names = [i for i in cPRC1_mcool_names if "UNT" in i]

for mcool1, name1 in zip(unt_files, unt_names):
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

    for mcool, name in zip([mcool1, mcool2], [name1, name2]):
        region_str1 = foxa2_loc
        region_str2 = insm1_loc
        resolution = 2000


        frame1 = XAxis()
        frame2 = XAxis()
        sub_frames = {"left": frame2, "bottom": frame1}

        heatmap = Cool(mcool, resolution=resolution, min_value = -4.5, color_bar = True, **ZOOM_PLOTTING_PARAMS)
        jv = JointView(heatmap)

        fig = jv.plot(region_str1, region_str2)

        fig.save(os.path.join(plot_dir, f"Insm1Foxa2_zoom_" + name + "cbar.svg"))


### Fig S7 example plots

In [ ]:
matches = ["dTAGPHC123CBX27dTAG_"]

cPRC1_mcool_files = sorted([mcool for mcool in mcool_files if any("/" + x in mcool for x in matches)])
cPRC1_mcool_names = sorted([name for name in mcool_names if name.startswith(tuple(matches))])

#For these ones, there isn't really any corresponding ChIP, so just plot the PRC1/2deg RING1B and SUZ12 tracks
chip_matches = ["RING1B_MERGED", "SUZ12_MERGED"]
matched_bigwigs_files = sorted([bw for bw in chip_bigwigs if "RING1AB" in bw and any(x in bw for x in chip_matches)])
matched_bigwigs_names = sorted([bw for bw in chip_bigwigs_names if "RING1AB" in bw and any(x in bw for x in chip_matches)])
#Note that sorted treats uppercase letters as before lowercase letters

matched_bigwigs_files


In [ ]:
RNA_forward_bigwigs = glob.glob(f'{RNA_path}*FORWARD*')
RNA_reverse_bigwigs = glob.glob(f'{RNA_path}*REVERSE*')
RNA_names = [Path(bigwig).stem.split('_FORWARD')[0] for bigwig in RNA_forward_bigwigs]

cPRC1_RNA_names = sorted([bw for bw in RNA_names if bw.startswith(tuple(matches))])
cPRC1_RNA_F = sorted([bw for bw in RNA_forward_bigwigs if any("/" + x in bw for x in matches)])
cPRC1_RNA_R = sorted([bw for bw in RNA_reverse_bigwigs if any("/" + x in bw for x in matches)])
cPRC1_RNA_F

#### Plot region 2 (Nkx2-2)

#### Region 2

In [ ]:
region_num = 1
RCMC_res = 5000

In [ ]:
region = cap_regs.iloc[region_num]
region_str = f'{region[0]}:{region[1]}-{region[2]}' # provide region in format like chr15:99,921,434-100,682,822

#RCMC
for cell_line in matches:
    unt_files = [i for i in cPRC1_mcool_files if "UNT" in i and cell_line in i]
    unt_names = [i for i in cPRC1_mcool_names if "UNT" in i and cell_line in i]
    filenum = 0
    mcool1 = unt_files[filenum]
    name1 = unt_names[filenum]
    if "UNT" in name1:
        mcool2 = mcool1.replace("UNT","dTAG")
        name2 = name1.replace("UNT","dTAG")

    microc_file1 = mcool1
    microc_file2 = mcool2
    microc_title = f"{name1} vs {name2} region {region_num + 1} at {RCMC_res}"

    heatmap = Cool(microc_file1, file_bottom=microc_file2, 
                   resolution=RCMC_res, **MICROC_PLOTTING_PARAMS) + Title(microc_title)
    frame = heatmap
    frame += XAxis()

    UNT_chip_bigwigs_names = [bigwigname for bigwigname in matched_bigwigs_names if "UNT" in bigwigname]
    UNT_chip_list = [chip for chip in matched_bigwigs_files if "UNT" in chip]

    bigwig_bins = 3600
    #Only plot the WT
    i = 0
    for UNT_chip, UNT_chip_bigwigs_name in zip(UNT_chip_list, UNT_chip_bigwigs_names):
        bigwig_list = make_bigwig_list([UNT_chip], region_str, [UNT_chip_bigwigs_name], 
                                   DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=bigwig_bins, autoscale=True)
        i += 1
        for bigwig in bigwig_list:
            frame += bigwig + Spacer(SPACER_HEIGHT)

    pro_bins = 3600

    forward_RNA_list = sorted([bw for bw in RNA_forward_bigwigs if "/" + cell_line in bw])
    print(forward_RNA_list)
    reverse_RNA_list = sorted([bw for bw in RNA_reverse_bigwigs if "/" + cell_line in bw])
    print(reverse_RNA_list)
    RNA_name_list = sorted([bw for bw in RNA_names if bw.startswith(cell_line)])
    print(RNA_name_list)

    f_bigwig_list = make_bigwig_list(forward_RNA_list, region_str, RNA_name_list, 
                                   DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=pro_bins,  
                                     track_height=PRO_SEQ_TRACK_HEIGHT, autoscale=True)
    r_bigwig_list = make_bigwig_list(reverse_RNA_list, region_str, RNA_name_list, 
                           DEFAULT_COLOR_LIST[(i+1)%len(DEFAULT_COLOR_LIST)], bw_bins=pro_bins, 
                                     track_height=PRO_SEQ_TRACK_HEIGHT, autoscale=True)
    r_bigwig_list = [r_bigwig + Inverted() for r_bigwig in r_bigwig_list]
    for f_bigwig, r_bigwig in zip(f_bigwig_list, r_bigwig_list):
        f_max_y = f_bigwig.properties['max_value']
        r_max_y = r_bigwig.properties['max_value']
        #Apply a consistent scaling factor
        #Divide this factor by 5 to allow the Polycomb-repressed genes to be seen
        max_y = round(max(f_max_y, r_max_y) / 5)
        f_bigwig_maxnorm = f_bigwig + MaxValue(max_y)
        r_bigwig_maxnorm = r_bigwig + MaxValue(max_y)

        frame += f_bigwig + r_bigwig + Spacer(SPACER_HEIGHT)

    annotations = NewBed(*ANNOTATIONS_PARAMS)
    frame += annotations

    fig = frame.plot(region_str)
    fig.savefig(os.path.join(plot_dir, f"region{region_num + 1}_{name1} over {name2} at {RCMC_res} bp TEMP" + ".pdf"), format='pdf')
    fig



#### Fig. S7 Plot zooms of specific interactions

Pax1-Foxa2

In [ ]:
pax1_loc = "chr2:147,162,000-147,262,000"
foxa2_loc = "chr2:147,832,000-147,942,000"

In [ ]:
unt_files = [i for i in cPRC1_mcool_files if "UNT" in i]
unt_names = [i for i in cPRC1_mcool_names if "UNT" in i]

for mcool1, name1 in zip(unt_files, unt_names):
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

    for mcool, name in zip([mcool1, mcool2], [name1, name2]):
        region_str1 = foxa2_loc
        region_str2 = pax1_loc
        resolution = 2000


        frame1 = XAxis()
        frame2 = XAxis()
        sub_frames = {"left": frame2, "bottom": frame1}

        heatmap = Cool(mcool, resolution=resolution, min_value = -4.5, color_bar = False, **MICROC_PLOTTING_PARAMS)
        jv = JointView(heatmap)

        fig = jv.plot(region_str1, region_str2)

        fig.save(os.path.join(plot_dir, f"Pax1Foxa2_zoom_" + name + ".svg"))


Insm1-Foxa2

In [ ]:
insm1_loc = "chr2:146,009,000-146,119,000"
foxa2_loc = "chr2:147,832,000-147,942,000"

In [ ]:
ZOOM_PLOTTING_PARAMS = {"style":"matrix",
                          "transform":"log10",
                          "depth_ratio":"full",
                          "balance":True,
                          "cmap":"fall",
                          "max_value":-2.5}

In [ ]:
unt_files = [i for i in cPRC1_mcool_files if "UNT" in i]
unt_names = [i for i in cPRC1_mcool_names if "UNT" in i]

for mcool1, name1 in zip(unt_files, unt_names):
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

    for mcool, name in zip([mcool1, mcool2], [name1, name2]):
        region_str1 = foxa2_loc
        region_str2 = insm1_loc
        resolution = 2000


        frame1 = XAxis()
        frame2 = XAxis()
        sub_frames = {"left": frame2, "bottom": frame1}

        heatmap = Cool(mcool, resolution=resolution, min_value = -4.5, color_bar = False, **ZOOM_PLOTTING_PARAMS)
        jv = JointView(heatmap)

        fig = jv.plot(region_str1, region_str2)

        fig.save(os.path.join(plot_dir, f"Insm1Foxa2_zoom_" + name + ".svg"))


Tbx5-Lhx5

In [ ]:
tbx5_loc = "chr5:119,900,000-120,000,000"
lhx5_loc = "chr5:120,521,000-120,621,000"


In [ ]:
ZOOM_PLOTTING_PARAMS = {"style":"matrix",
                          "transform":"log10",
                          "depth_ratio":"full",
                          "balance":True,
                          "cmap":"fall",
                          "max_value":-2}

In [ ]:
unt_files = [i for i in PRC12_mcool_files if "UNT" in i]
unt_names = [i for i in PRC12_mcool_names if "UNT" in i]
filenum = 0
mcool1 = unt_files[filenum]
name1 = unt_names[filenum]
if "UNT" in name1:
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

for mcool, name in zip([mcool1, mcool2], [name1, name2]):
    region_str1 = lhx5_loc
    region_str2 = tbx5_loc
    resolution = 2000


    frame1 = XAxis()
    frame2 = XAxis()
    sub_frames = {"left": frame2, "bottom": frame1}

    heatmap = Cool(mcool, resolution=resolution, min_value = -4.5, color_bar = False, **ZOOM_PLOTTING_PARAMS)
    # jv = JointView(heatmap, **sub_frames, space = 0, padding_left = 0)
    jv = JointView(heatmap)

    fig = jv.plot(region_str1, region_str2)
    
    fig.save(os.path.join(plot_dir, f"Tbx5Lhx5_zoom_" + name + "adj_contrast.svg"))


In [ ]:
unt_files = [i for i in PRC12_mcool_files if "UNT" in i]
unt_names = [i for i in PRC12_mcool_names if "UNT" in i]
filenum = 0
mcool1 = unt_files[filenum]
name1 = unt_names[filenum]
if "UNT" in name1:
    mcool2 = mcool1.replace("UNT","dTAG")
    name2 = name1.replace("UNT","dTAG")

for mcool, name in zip([mcool1, mcool2], [name1, name2]):
    region_str1 = lhx5_loc
    region_str2 = tbx5_loc
    resolution = 2000


    frame1 = XAxis()
    frame2 = XAxis()
    sub_frames = {"left": frame2, "bottom": frame1}

    heatmap = Cool(mcool, resolution=resolution, min_value = -4.5, color_bar = True, **ZOOM_PLOTTING_PARAMS)
    # jv = JointView(heatmap, **sub_frames, space = 0, padding_left = 0)
    jv = JointView(heatmap)

    fig = jv.plot(region_str1, region_str2)
    
    fig.save(os.path.join(plot_dir, f"Tbx5Lhx5_zoom_" + name + "adj_contrast_cbar.svg"))


### Fig 5 example plots

In [ ]:
matches = ["PCGF24dTAGdTAGSUZ12_", "dTAGPCGF1PCGF24dTAG_", "dTAGPCGF6PCGF24dTAG_"]

cPRC1_mcool_files = sorted([mcool for mcool in mcool_files if any("/" + x in mcool for x in matches)])
cPRC1_mcool_names = sorted([name for name in mcool_names if name.startswith(tuple(matches))])

#For these ones, there isn't really any corresponding ChIP, so just plot the PRC1/2deg RING1B and SUZ12 tracks
chip_matches = ["RING1B_MERGED", "SUZ12_MERGED"]
cPRC1_bigwigs_files = sorted([bw for bw in chip_bigwigs if "RING1AB" in bw and any(x in bw for x in chip_matches)])
cPRC1_bigwigs_names = sorted([bw for bw in chip_bigwigs_names if "RING1AB" in bw and any(x in bw for x in chip_matches)])
#Note that sorted treats uppercase letters as before lowercase letters

cPRC1_bigwigs_files


In [ ]:
#No TT-seq here because one cell line (PCGF2/4 SUZ12) doesn't have it

#### Plot region 2 (Nkx2-2)

#### Region 2

In [ ]:
region_num = 1
RCMC_res = 5000

In [ ]:
region = cap_regs.iloc[region_num]
region_str = f'{region[0]}:{region[1]}-{region[2]}' # provide region in format like chr15:99,921,434-100,682,822

#RCMC
for cell_line in matches:
    unt_files = [i for i in cPRC1_mcool_files if "UNT" in i and cell_line in i]
    unt_names = [i for i in cPRC1_mcool_names if "UNT" in i and cell_line in i]
    filenum = 0
    mcool1 = unt_files[filenum]
    name1 = unt_names[filenum]
    if "UNT" in name1:
        mcool2 = mcool1.replace("UNT","dTAG")
        name2 = name1.replace("UNT","dTAG")

    microc_file1 = mcool1
    microc_file2 = mcool2
    microc_title = f"{name1} vs {name2} region {region_num + 1} at {RCMC_res}"

    heatmap = Cool(microc_file1, file_bottom=microc_file2, 
                   resolution=RCMC_res, **MICROC_PLOTTING_PARAMS) + Title(microc_title)
    frame = heatmap
    frame += XAxis()

    UNT_chip_bigwigs_names = [bigwigname for bigwigname in cPRC1_bigwigs_names if "UNT" in bigwigname]
    UNT_chip_list = [chip for chip in cPRC1_bigwigs_files if "UNT" in chip]

    bigwig_bins = 3600
    #Only plot the WT
    i = 0
    for UNT_chip, UNT_chip_bigwigs_name in zip(UNT_chip_list, UNT_chip_bigwigs_names):
        bigwig_list = make_bigwig_list([UNT_chip], region_str, [UNT_chip_bigwigs_name], 
                                   DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=bigwig_bins, autoscale=True)
        i += 1
        for bigwig in bigwig_list:
            frame += bigwig + Spacer(SPACER_HEIGHT)

    annotations = NewBed(*ANNOTATIONS_PARAMS)
    frame += annotations

    fig = frame.plot(region_str)
    fig.savefig(os.path.join(plot_dir, f"region{region_num + 1}_{name1} over {name2} at {RCMC_res} bp" + ".pdf"), format='pdf')
    fig



#### Fig. 5 Plot zooms of specific interactions

Pax1-Foxa2

In [ ]:
pax1_loc = "chr2:147,162,000-147,262,000"
foxa2_loc = "chr2:147,832,000-147,942,000"

In [ ]:

for cell_line in matches:
    unt_files = [i for i in cPRC1_mcool_files if "UNT" in i and cell_line in i]
    unt_names = [i for i in cPRC1_mcool_names if "UNT" in i and cell_line in i]
    #A for loop is not needed here...
    for mcool1, name1 in zip(unt_files, unt_names):
        mcool2 = mcool1.replace("UNT","dTAG")
        name2 = name1.replace("UNT","dTAG")
        print(name1 + " " + name2)
        print(mcool1 + " " + mcool2)

        for mcool, name in zip([mcool1, mcool2], [name1, name2]):
            region_str1 = foxa2_loc
            region_str2 = pax1_loc
            resolution = 2000


            frame1 = XAxis()
            frame2 = XAxis()
            sub_frames = {"left": frame2, "bottom": frame1}

            heatmap = Cool(mcool, resolution=resolution, min_value = -4.5, color_bar = False, **MICROC_PLOTTING_PARAMS)
            jv = JointView(heatmap)

            fig = jv.plot(region_str1, region_str2)

            fig.save(os.path.join(plot_dir, f"Pax1Foxa2_zoom_" + name + ".svg"))


Insm1-Foxa2

In [ ]:
insm1_loc = "chr2:146,009,000-146,119,000"
foxa2_loc = "chr2:147,832,000-147,942,000"

In [ ]:
ZOOM_PLOTTING_PARAMS = {"style":"matrix",
                          "transform":"log10",
                          "depth_ratio":"full",
                          "balance":True,
                          "cmap":"fall",
                          "max_value":-2.5}

In [ ]:
for cell_line in matches:
    unt_files = [i for i in cPRC1_mcool_files if "UNT" in i and cell_line in i]
    unt_names = [i for i in cPRC1_mcool_names if "UNT" in i and cell_line in i]
    #A for loop is not needed here, this could be written better...
    for mcool1, name1 in zip(unt_files, unt_names):
        mcool2 = mcool1.replace("UNT","dTAG")
        name2 = name1.replace("UNT","dTAG")
        print(name1 + " " + name2)
        print(mcool1 + " " + mcool2)
        for mcool1, name1 in zip(unt_files, unt_names):
            mcool2 = mcool1.replace("UNT","dTAG")
            name2 = name1.replace("UNT","dTAG")

            for mcool, name in zip([mcool1, mcool2], [name1, name2]):
                region_str1 = foxa2_loc
                region_str2 = insm1_loc
                resolution = 2000


                frame1 = XAxis()
                frame2 = XAxis()
                sub_frames = {"left": frame2, "bottom": frame1}

                heatmap = Cool(mcool, resolution=resolution, min_value = -4.5, color_bar = False, **ZOOM_PLOTTING_PARAMS)
                jv = JointView(heatmap)

                fig = jv.plot(region_str1, region_str2)

                fig.save(os.path.join(plot_dir, f"Insm1Foxa2_zoom_" + name + ".svg"))
